In [ ]:
!nvidia-smi


In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib
import random as pyrandom
from scipy import ndimage
from torch.utils.data import Dataset,DataLoader
import torchvision.transforms as T
from torchvision.transforms import Composeimport os

In [ ]:
def checkfilepath(path):
    return os.path.exists(str(path))

In [ ]:
df = pd.read_excel('path', engine='openpyxl')
df['SubjID'].replace('',np.nan,inplace=True)
df['AGE_at_scan'].replace('',np.nan,inplace=True)
df['DX'].replace('',np.nan,inplace=True)
df.dropna(subset=['SubjID','AGE_at_scan','DX'], inplace=True)
df.loc[df['SEX']=='M','SEX'] = 1
df.loc[df['SEX']=='F','SEX'] = 0
df.loc[df['DX']=='CN','DX'] = 0
df.loc[df['DX']=='MCI','DX'] = 1
df.loc[df['DX']=='Dementia','DX'] = 2
df = df.sort_values(by = ['SubjID'])
df = df.reset_index(drop=True)

In [ ]:
class Preprocess(object):
    def rotate(self,vol):
        def scipy_rotate(vol):
            vol[vol<0] = 0
            vol[vol>1] = 1
            return vol
        aug_vol = scipy_rotate(vol)
        return aug_vol
    
    def __call__(self,sample):
        sample = self.rotate(sample)
        sample = np.expand_dims(sample,0)
        return torch.from_numpy(sample)
transformX = Preprocess()

In [ ]:
class MRIDataset(torch.utils.data.Dataset):
    """
    The custom MRI Dataset
    """
    def read_scan(self,path):
        scan = nib.load(path)
        volume = scan.get_fdata()
        min = np.amax(volume)
        max = np.amin(volume)
        volume = (volume - min) / (max - min)
        volume = volume.astype("float32")
        return volume

    def __init__(self, data,p = None,transform=None,sample_weights=None):
        self.data = data
        self.p = p
        self.sample_weights = sample_weights
        self.transform=transform
        
    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        t1  = self.read_scan(self.data['T1'].iloc[idx])
        dx = self.data['DX'].iloc[idx]
        x = torch.rand(1).item()
        if((self.p is not None) and (x<self.p)):
            dx = -1
        dx = np.asarray([dx],dtype='float32')
        dx = np.expand_dims(dx,-1)
        if self.transform:
            t1 = self.transform(t1)
        return {'mri':t1,'label':dx}

In [ ]:
dg_train = MRIDataset(scans,p=0.10,transform=transformX)
dl_train = DataLoader(dg_train,batch_size=1,shuffle=True)

In [ ]:
import os
import tempfile
import time
import torch.nn.functional as F
import monai
from monai.config import print_config
from monai.data import DataLoader
from monai.transforms import (
    EnsureChannelFirstd,
    CenterSpatialCropd,
    Compose,
    Lambdad,
    LoadImaged,
    Resized,
    ScaleIntensityd
)
from monai.utils import set_determinism
from torch.cuda.amp import GradScaler, autocast
from tqdm import tqdm

from generative.inferers import DiffusionInferer
from generative.networks.nets import DiffusionModelUNet
from generative.networks.schedulers import DDPMScheduler, DDIMScheduler

print_config()

In [ ]:
device = torch.device('cuda:0')
model = DiffusionModelUNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=1,
    num_channels=(128, 256, 512),
    attention_levels=(False, False, True),
    num_res_blocks=2,
    num_head_channels=(0, 0, 512),
    with_conditioning=True,
    cross_attention_dim=1,
)
model.to(device)

In [ ]:
scheduler = DDPMScheduler(num_train_timesteps=1000, schedule="scaled_linear_beta", beta_start=0.0005, beta_end=0.0195)

In [ ]:
inferer = DiffusionInferer(scheduler)
optimizer = torch.optim.Adam(params=model.parameters(), lr=5e-5)

In [ ]:
n_epochs = 150
epoch_loss_list = []
val_epoch_loss_list = []

scaler = GradScaler()
total_start = time.time()
for epoch in range(n_epochs):
    model.train()
    epoch_loss = 0
    progress_bar = tqdm(enumerate(dl_train), total=len(dl_train), ncols=70)
    progress_bar.set_description(f"Epoch {epoch}")
    for step, batch in progress_bar:
        images = batch['mri'].to(device)
        classes = batch['label'].to(device)
        optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=True):
            noise = torch.randn_like(images).to(device)
            timesteps = torch.randint(
                0, inferer.scheduler.num_train_timesteps, (images.shape[0],), device=images.device
            ).long()

            # Get model prediction
            noise_pred = inferer(inputs=images, diffusion_model=model, noise=noise, timesteps=timesteps,condition=classes)

            loss = F.mse_loss(noise_pred.float(), noise.float())

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()

        progress_bar.set_postfix({"loss": epoch_loss / (step + 1)})
    epoch_loss_list.append(epoch_loss / (step + 1))
    
total_time = time.time() - total_start
print(f"train completed, total time: {total_time}.")

In [ ]:
model.eval()
guidance_scale = 7.0
conditioning = torch.cat([-1 * torch.ones(1, 1, 1).float(), 1*torch.ones(1, 1, 1).float()], dim=0).to(device)

noise = torch.randn((1, 1, 91, 109, 91))
noise = noise.to(device)
scheduler.set_timesteps(num_inference_steps=1000)
progress_bar = tqdm(scheduler.timesteps)
for t in progress_bar:
    with autocast(enabled=True):
        with torch.no_grad():
            noise_input = torch.cat([noise] * 2)
            model_output = model(noise_input, timesteps=torch.Tensor((t,)).to(noise.device), context=conditioning)
            noise_pred_uncond, noise_pred_text = model_output.chunk(2)
            noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)

    noise, _ = scheduler.step(noise_pred, t, noise)

In [ ]:
## SOME EVALUATION CODE

for idx in range(0,300):
    model.eval()
    guidance_scale = 7.0
    cn = 0
    if(idx%3==0): cn = 0
    elif(idx%3==1):cn=1
    elif(idx%3==2):cn=2
    conditioning = torch.cat([-1 * torch.ones(1, 1, 1).float(), cn*torch.ones(1, 1, 1).float()], dim=0).to(device)
    
    noise = torch.randn((1, 1, 91, 109, 91))
    noise = noise.to(device)
    scheduler.set_timesteps(num_inference_steps=1000)
    progress_bar = tqdm(scheduler.timesteps)
    for t in progress_bar:
        with autocast(enabled=True):
            with torch.no_grad():
                noise_input = torch.cat([noise] * 2)
                model_output = model(noise_input, timesteps=torch.Tensor((t,)).to(noise.device), context=conditioning)
                noise_pred_uncond, noise_pred_text = model_output.chunk(2)
                noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)
    
        noise, _ = scheduler.step(noise_pred, t, noise)
    gen_img = noise.cpu().numpy()
    if(idx%3==0):
        np.save(f'path/cn/gen_image_cn_{idx//3}.npy',gen_img)
    elif(idx%3==1):
        np.save(f'path/mci/gen_image_ad_{(idx-1)//3}.npy',gen_img)
    elif(idx%3==2):
        np.save(f'path/ad/gen_image_ad_{(idx-2)//3}.npy',gen_img)